# Lab 08: Engineering and Scale: The Teacher Server, Measured

**Tier 2 lab.** Part A executes and asserts anywhere; Part B runs only when `RUN_SERVER = True`
on the training box.

**The question.** Every lab so far co-located teacher and student in one process, which stops
working the moment the teacher outgrows its co-tenant: a 20-32B-class teacher wants its own
memory pool, its own batching engine, and its own lifecycle. The standard answer is a
**teacher server**: vLLM (a dedicated inference engine) serves the teacher over HTTP, while
the student trains in a separate process and sends the tokens it wants scored across that
HTTP connection. TRL's `DistillationTrainer` supports exactly this topology
(`vllm_server_base_url` + mode `"server"`).

**The second question is the lab's real subject: what are this machine's actual numbers?**
This course has quoted "~2,000 tok/s prefill, ~50 tok/s decode, ~40:1" for a 20B-class model
since the README, where tok/s is tokens per second, prefill is the parallel scoring pass over
existing tokens, and decode is token-by-token generation. Those numbers came from someone
else's benchmark of this hardware class. Part B replaces them with **your own measured
curves**, throughput versus batch size versus sequence length, on your box, with your
teacher. The curves are written to `machine_profile.json`, which Labs 09-12 then treat as
ground truth for their sizing. The quoted numbers become the *expected range* your
measurement is checked against, which is the correct final relationship between a course and
its reader: the course's claims end as your data's null hypothesis.

A server topology also buys three things co-location cannot. First, **continuous batching**
across asynchronous scoring requests: the server does not wait for a fixed batch to fill, it
keeps packing whatever requests are in flight into dense work, so your ragged, mixed-length
rollouts get packed into dense prefill. Second, independent restart: a wedged teacher no
longer kills your optimizer state, because the teacher lives in its own process. Third,
quantized serving: quantization stores weights in fewer bytes per parameter (4-bit means half
a byte per parameter instead of two bytes for bf16), so a 32B teacher in 4-bit fits alongside
a full training run. The topology costs one thing: the network hop returns **top-k logprobs,
not full logits**. A logprobs payload gives you the log-probabilities of only the k most
likely tokens at each position, not the model's full scores over the whole vocabulary. That
is why Lab 02 §5's top-k machinery was built before this lab, and why the remote loss below
is `topk_forward_kl` against a server-built cache batch.

In [1]:
import sys, os, json, math, time, threading
sys.path.insert(0, "../code")

import numpy as np
import torch
import torch.nn.functional as F

from kd_core import topk_forward_kl, bytes_per_token_cache
from kd_pipeline import (set_seed_everywhere, MemoryPlan, infer_gb, full_ft_gb,
                         kv_cache_gb, bandwidth_bound_decode_tps, RunManifest)

RUN_SERVER = False        # <-- flip on the training box (starts/uses a vLLM server)
SEED = 17
set_seed_everywhere(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device} | RUN_SERVER: {RUN_SERVER}")

torch 2.13.0+cpu | device: cpu | RUN_SERVER: False


## Part A · 1: The roofline, and what it can and cannot promise

A reminder of why decode has a hard ceiling. Each generated token requires one full forward
pass, and each pass streams the model's working set (its weights) from memory once, so decode
speed is capped by memory bandwidth divided by the bytes read per token. That ceiling is the
**roofline**, and `bandwidth_bound_decode_tps` computes it as an **upper bound**: decode
cannot beat bandwidth ÷ bytes-per-token. There are two disciplined uses of this bound, and
one abuse:

- *Use 1: a sanity check on measurements.* Any measured decode rate **above** the roofline
  for the dense weight footprint means one of two things. Either you mis-measured, or the
  model is MoE or quantized and your footprint number is wrong. The roofline is really about
  *bytes read per token*. For a MoE model (mixture of experts: the model holds many expert
  weight blocks but routes each token through only a few, so only the "active parameters" are
  read per token), the bytes read per token come from the active-expert slice, not from the
  checkpoint size on disk.
- *Use 2: design ordering.* Ratios between configurations (Lab 06's table) survive even when
  absolute predictions are loose.
- *The abuse: treating it as a prediction.* Real decode lands **well under** the bound,
  because the weights are not the only cost: attention computation, KV cache reads, kernel
  overheads, and scheduling all take time the bound ignores.

Here is the worked case that makes Use 1 concrete. The published 20B-MXFP4 measurement is
~49.7 tok/s (MXFP4 is a 4-bit quantization format, about half a byte per parameter). Its
weight footprint is ~10 GB, and if decode had to read all 10 GB per token the roofline would
be 273/10 ≈ 27 tok/s. But 49.7 was measured, which is *above* that ceiling. A measurement
above a hard bound is a contradiction, so one of the inputs must be wrong, and it is the
footprint: MXFP4 GPT-OSS is **MoE**, so its bytes-per-token is the active slice. Work it:
~3.6B active params × 0.5 bytes each ≈ 1.8 GB read per token, so the real bound is
273/1.8 ≈ 150 tok/s, and 49.7 sits comfortably under *that*. The roofline taught us to read
the architecture before trusting a footprint number. That reasoning chain is asserted below.

With it comes the KV-cache arithmetic that decides your server's `max_num_seqs` (the maximum
number of sequences it serves at once). The KV cache is the store of per-token attention keys
and values the model keeps so it does not recompute them for every new token; it grows
linearly with sequence length and with the number of concurrent sequences, so it is the
number that actually sizes a server.

In [2]:
BW = 273.0

# The MoE lesson, as arithmetic: the dense-footprint bound would be violated,
# the active-slice bound is respected. Measurement + roofline caught the architecture.
measured_20b_decode = 49.7
dense_bound  = bandwidth_bound_decode_tps(10.0, BW, bytes_per_param=1.0)   # 10 GB read/token IF dense
active_bound = bandwidth_bound_decode_tps(3.6,  BW, bytes_per_param=0.5)   # MoE active slice, 4-bit
print(f"measured decode          : {measured_20b_decode:6.1f} tok/s")
print(f"bound if dense 10 GB/tok : {dense_bound:6.1f} tok/s  -> violated, so NOT dense reads")
print(f"bound for MoE active slice: {active_bound:6.1f} tok/s  -> respected")
assert measured_20b_decode > dense_bound, "the contradiction that exposes the MoE"
assert measured_20b_decode < active_bound, "the active-slice roofline must hold"

# KV cache: the number that actually sizes a server. Qwen3-32B-class geometry (GQA):
n_layers, n_kv_heads, head_dim = 64, 8, 128
print(f"\nKV per sequence, 32B-class GQA geometry ({n_layers}L x {n_kv_heads}kv x {head_dim}):")
for seq in (2048, 8192, 32768):
    per_seq = kv_cache_gb(n_layers, n_kv_heads, head_dim, seq, batch=1)
    max_seqs = int((128 - infer_gb(32, 0.55) - 8) / per_seq)   # 4-bit-ish teacher + overhead
    print(f"  seq {seq:>6}: {per_seq*1e3:7.1f} MB/seq  -> ~{max_seqs} concurrent seqs in the leftover")
assert kv_cache_gb(n_layers, n_kv_heads, head_dim, 32768, 1) > 10 * \
       kv_cache_gb(n_layers, n_kv_heads, head_dim, 2048, 1), "KV grows linearly in seq"
print("\nroofline used as a bound and a lie-detector, never as a prediction")

measured decode          :   49.7 tok/s
bound if dense 10 GB/tok :   27.3 tok/s  -> violated, so NOT dense reads
bound for MoE active slice:  151.7 tok/s  -> respected

KV per sequence, 32B-class GQA geometry (64L x 8kv x 128):
  seq   2048:   536.9 MB/seq  -> ~190 concurrent seqs in the leftover
  seq   8192:  2147.5 MB/seq  -> ~47 concurrent seqs in the leftover
  seq  32768:  8589.9 MB/seq  -> ~11 concurrent seqs in the leftover

roofline used as a bound and a lie-detector, never as a prediction


## Part A · 2: The remote-scoring client, tested against a mock server

The client speaks vLLM's OpenAI-compatible `/v1/completions` endpoint. "OpenAI-compatible"
means the server copies the HTTP request and response shapes of OpenAI's public API, so any
client written for that API works against it. On top of that endpoint sits the
`prompt_logprobs` extension: you send token ids, and you get back, for each position in the
prompt, the log-probabilities of the k most likely tokens. That per-position top-k logprobs
payload is exactly the payload `TopKCacheReader.batch` shapes for `topk_forward_kl`, which
means remote scoring reuses the entire Lab 04 loss path unchanged. That reuse is the design
working as intended: the objective code never learns whether the teacher was local, cached,
or a URL.

Testing a network client requires a server, so this cell *runs one*: a mock that speaks the
vLLM response shape, running in a thread for exactly as long as the test needs. The client is
then verified end-to-end: correct parse, correct tensor shapes, and correct loss against a
locally computed reference built from the same logits the mock serves. When Part B swaps the
URL to a real vLLM, every line of client code has already run.

In [3]:
import http.server, socketserver, requests

V_mock, K_mock, T_mock = 128, 8, 6
g = torch.Generator().manual_seed(0)
MOCK_LOGITS = 5 * torch.randn(1, T_mock, V_mock, generator=g)   # the mock's "teacher"

class MockVLLM(http.server.BaseHTTPRequestHandler):
    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        k = body["prompt_logprobs"]
        lp = F.log_softmax(MOCK_LOGITS.float(), -1)
        top_lp, top_idx = lp.topk(k, dim=-1)
        # vLLM shape: prompt_logprobs[pos] = {token_id_str: {"logprob": ...}}; pos 0 is null
        pl = [None] + [
            {str(int(top_idx[0, t, j])): {"logprob": float(top_lp[0, t, j])}
             for j in range(k)}
            for t in range(1, T_mock)]
        out = {"choices": [{"prompt_logprobs": pl}]}
        self.send_response(200); self.send_header("Content-Type", "application/json")
        self.end_headers(); self.wfile.write(json.dumps(out).encode())
    def log_message(self, *a):  # quiet
        pass

def score_remote(base_url, model, token_ids, k):
    '''Returns a cache-style batch for topk_forward_kl, positions 1..T-1.'''
    resp = requests.post(f"{base_url}/v1/completions", json={
        "model": model, "prompt": token_ids, "max_tokens": 1, "prompt_logprobs": k,
    }, timeout=120).json()
    pl = resp["choices"][0]["prompt_logprobs"][1:]      # drop the null pos-0
    T = len(pl)
    lp = torch.full((1, T, k), -30.0); idx = torch.zeros(1, T, k, dtype=torch.long)
    for t, d in enumerate(pl):
        for j, (tid, rec) in enumerate(sorted(d.items(),
                                              key=lambda kv: -kv[1]["logprob"])[:k]):
            lp[0, t, j] = rec["logprob"]; idx[0, t, j] = int(tid)
    tail = (1 - lp.exp().sum(-1)).clamp_min(1e-9).log()
    return {"topk_logprobs": lp, "topk_idx": idx, "tail_logprob": tail,
            "k": torch.tensor(k)}

with socketserver.TCPServer(("127.0.0.1", 0), MockVLLM) as srv:
    port = srv.server_address[1]
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    cache = score_remote(f"http://127.0.0.1:{port}", "mock", list(range(T_mock)), K_mock)
    srv.shutdown()

# End-to-end check: remote loss == local loss from the same logits.
s_logits = torch.randn(1, T_mock - 1, V_mock, generator=g)
m = torch.ones(1, T_mock - 1, dtype=torch.bool)
remote_kl = topk_forward_kl(s_logits, cache, m, scale_by_T2=False)
from kd_core import make_topk_cache
local_kl = topk_forward_kl(s_logits, {k2: (v[:, 1:] if v.dim() > 1 else v) for k2, v in
                                      make_topk_cache(MOCK_LOGITS, k=K_mock).items()},
                           m, scale_by_T2=False)
assert abs(float(remote_kl) - float(local_kl)) < 1e-4, (float(remote_kl), float(local_kl))
print(f"remote-scored KL {float(remote_kl):.5f} == locally computed {float(local_kl):.5f}")
print("client verified end-to-end against a live (mock) server — Part B changes only the URL")

remote-scored KL 4.21866 == locally computed 4.21866
client verified end-to-end against a live (mock) server — Part B changes only the URL


## Part A · 3: The benchmark harness and the profile schema

`machine_profile.json` is this lab's artifact, the file later labs cite instead of the
README's borrowed numbers. The schema: for each (phase, batch, seq) cell, a tokens-per-second
number and the timing method that produced it. The harness itself is the thing most often
gotten wrong in amateur benchmarks, so its three rules are encoded and unit-tested against a
fake workload with a *known* rate. Rule one: warmup iterations are discarded, because the
first few runs pay one-time costs (compilation, caches filling) that do not represent steady
state. Rule two: report the median of N timed runs, not the mean of one, so a single noisy
run cannot skew the result. Rule three, on CUDA: explicitly synchronize before stopping the
clock, because GPU kernel launches return immediately while the work is still running, so
without a synchronize you time the launch rather than the work.

In [4]:
def bench(fn, n_warmup=2, n_timed=5):
    '''Median wall-clock of fn() with warmup; fn returns tokens processed.'''
    for _ in range(n_warmup):
        fn()
    times, toks = [], None
    for _ in range(n_timed):
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        toks = fn()
        if device == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return toks / sorted(times)[len(times) // 2]

# Unit test with a known-rate fake: 5,000 "tokens" in ~50 ms -> ~100k tok/s.
def fake_workload():
    time.sleep(0.05)
    return 5_000
rate = bench(fake_workload, n_warmup=1, n_timed=3)
assert 60_000 < rate < 110_000, f"harness mis-measures a known workload: {rate:,.0f}"
print(f"harness verified on a known workload: {rate:,.0f} tok/s (true ~100k)")

PROFILE_SCHEMA = {
    "hardware_note": "single box, 128 GB unified, ~273 GB/s, arm64/sm_121",
    "teacher": None, "quantization": None, "server": "vllm",
    "prefill": {},      # f"{batch}x{seq}" -> tok/s
    "decode": {},       # f"{batch}x{new_tokens}" -> tok/s
    "measured_ratio_prefill_to_decode": None,
    "expected_range_note": "course prior: prefill ~2000, decode ~50, ratio ~40:1 (20B-class)",
}
print("profile schema ready; Part B fills it with measurements, not quotations")

harness verified on a known workload: 99,672 tok/s (true ~100k)
profile schema ready; Part B fills it with measurements, not quotations


## Part B: Serve, measure, train

Three phases, each restartable on its own. That independence is the topology's gift.

**B·1 Serve.** From a terminal on the training box (use a known-good aarch64 vLLM container;
do not build from source casually on this platform, because wheel availability is the
recurring friction):

```bash
vllm serve Qwen/Qwen3-32B-AWQ \
    --max-model-len 4096 --gpu-memory-utilization 0.55 \
    --max-num-seqs 16 --port 8000
```

The model name's AWQ suffix means the weights are quantized with AWQ, a 4-bit format, so each
parameter costs about half a byte instead of bf16's two bytes; that is what lets a 32B
teacher share the box with a training run. The `0.55` is not caution, it is the co-tenancy
budget from A·1: the student's training process needs its share of the same unified memory.
Verify with a one-line `requests.get(...)/v1/models` before anything else.

**B·2 Measure.** Prefill and decode curves over batch × length, written to the profile.
Watch for the two shapes worth knowing. First: prefill tokens/sec *rising* with batch until
compute saturates, because larger batches keep the arithmetic units busier until they are
fully occupied. Second: decode tokens/sec in aggregate rising with batch **while per-stream
latency falls**, meaning each individual sequence generates more slowly even as the total
across sequences grows. That is continuous batching (the server packing all in-flight
requests into shared work) trading latency for throughput: one read of the weights now serves
every sequence in the batch.

**B·3 Train against the server.** The Lab 04 student, the Lab 04 loss, the A·2 client, and
the teacher is now just a URL. Log the fraction of step time spent waiting on scoring,
because that number decides whether your next run wants a bigger scoring batch or a colocated
teacher.

In [5]:
SERVER = "http://127.0.0.1:8000"
TEACHER_SERVED = "Qwen/Qwen3-32B-AWQ"

def measure_prefill(profile, batches=(1, 4, 16), seqs=(512, 2048)):
    for b in batches:
        for s in seqs:
            ids = np.random.randint(1000, 20000, size=(b, s)).tolist()
            def one():
                for row in ids:
                    requests.post(f"{SERVER}/v1/completions", json={
                        "model": TEACHER_SERVED, "prompt": row,
                        "max_tokens": 1, "prompt_logprobs": 1}, timeout=300)
                return b * s
            profile["prefill"][f"{b}x{s}"] = round(bench(one, 1, 3), 1)
            print(f"prefill {b}x{s}: {profile['prefill'][f'{b}x{s}']:>10,.0f} tok/s")

def measure_decode(profile, batches=(1, 8), new_tokens=256):
    for b in batches:
        prompts = [[1000 + i] * 32 for i in range(b)]
        def one():
            for p in prompts:
                requests.post(f"{SERVER}/v1/completions", json={
                    "model": TEACHER_SERVED, "prompt": p,
                    "max_tokens": new_tokens}, timeout=600)
            return b * new_tokens
        profile["decode"][f"{b}x{new_tokens}"] = round(bench(one, 1, 3), 1)
        print(f"decode  {b}x{new_tokens}: {profile['decode'][f'{b}x{new_tokens}']:>10,.1f} tok/s")

if RUN_SERVER:
    assert requests.get(f"{SERVER}/v1/models", timeout=10).ok, "server not up"
    profile = dict(PROFILE_SCHEMA, teacher=TEACHER_SERVED, quantization="AWQ-4bit")
    measure_prefill(profile)
    measure_decode(profile)
    pf = max(profile["prefill"].values()); dc = max(profile["decode"].values())
    profile["measured_ratio_prefill_to_decode"] = round(pf / dc, 1)
    os.makedirs("../runs/lab08", exist_ok=True)
    with open("../runs/lab08/machine_profile.json", "w") as f:
        json.dump(profile, f, indent=2)
    print(f"\nYOUR ratio: {profile['measured_ratio_prefill_to_decode']}:1 "
          f"(course prior said ~40:1) — machine_profile.json written")
    # B·3: swap Lab 04's stage-2 teacher for score_remote(SERVER, ...) and train;
    # the loss path is identical by construction (A·2). Log time-in-scoring per step.
else:
    print("RUN_SERVER=False — Part B compiled but did not execute.")
    print("Bring up the server, rerun, and retire this course's borrowed numbers.")

RUN_SERVER=False — Part B compiled but did not execute.
Bring up the server, rerun, and retire this course's borrowed numbers.


## Part C: The verdict

**Reading your curves.**

- *Prefill vs batch:* rising then flat. The flat region's height is your true corpus-caching
  rate; use it to reprice Lab 04's stage 1 and Lab 06's table with your numbers.
- *Decode vs batch:* aggregate rises while per-stream falls, as B·2 explained. The crossing
  point where the aggregate stops improving is your server's practical `max_num_seqs`.
  Compare it to the KV-cache arithmetic from A·1; the two should agree within ~2×, and the
  direction of the miss tells you which resource runs out first, attention compute or KV
  bandwidth (the memory traffic of reading the cached keys and values every step).
- *The ratio:* expect the measured prefill:decode ratio for a dense-serving teacher in the
  **20-60:1** band. Under 10:1 means prefill is being throttled (check
  `max_num_batched_tokens`); over 100:1 means decode is starved (check quantization kernel
  availability on this platform, because a quantized model still needs fast kernels for its
  format to decode at full speed).

**Failure signatures.**

- *Server OOM (out of memory) on long prompts, fine on short.* KV, always KV: the cache
  grows with sequence length, so long prompts are exactly where it blows past the budget.
  A·1's per-sequence MB × your actual concurrency exceeded the leftover memory; lower
  `--max-model-len` or `--max-num-seqs`.
- *Throughput collapses when training starts.* The two processes are fighting over the same
  memory bandwidth, which is expected on unified memory, where server and trainer draw from
  one pool. The mitigation is bigger, less frequent scoring batches, which amortizes the
  contention over more useful work per fight. Do not shrink the student batch; that is not
  where the contention comes from.
- *Remote KL slightly but systematically above local KL for the same pairs.* Top-k truncation
  at the server's `k`: the payload only carries the k most likely tokens, and the missing
  tail biases the estimate. This is Lab 02 §5's renormalise-vs-tail bias, now live. Raise
  `prompt_logprobs` or accept the measured bias; either is fine, *unmeasured* is not.
- *Scoring latency dominates step time.* Batch the scoring calls across
  gradient-accumulation microbatches (the small batches whose gradients are summed into one
  optimizer step): send one scoring request per optimizer step, not one per microbatch,
  before considering topology changes.

**The verdict to write:** three numbers with provenance (your peak prefill, your peak
aggregate decode, your ratio) and one sentence on where they diverge from the course's priors
and why you believe your measurement. From here on, *your* profile is the authority the
remaining labs cite.

## Exercises

1. **Quantization ladder.** Serve the same teacher in bf16 (if it fits alone), 8-bit, and
   4-bit; measure the decode curve for each. Does throughput scale with bytes-per-param the
   way the roofline predicts, or does kernel quality dominate on this platform?
2. **The co-tenancy frontier.** Sweep `--gpu-memory-utilization` ∈ {0.4, 0.55, 0.7} while a
   fixed student training job runs. Plot both processes' throughput; find the allocation that
   maximizes *joint* progress. That number is your box's real teacher-budget.
3. **Score-batch amortization.** Measure student steps/sec at scoring batch ∈ {1, 4, 16}
   rollouts per request. Where does the curve knee? Reconcile with the prefill curve from B·2.
4. **MoE audit.** Serve a small MoE model, measure decode, and back out its effective
   bytes-per-token from the roofline. Compare to `n_active_params × bytes_per_param`. You
   have now measured "dark bandwidth", the bytes actually read per token as revealed by
   throughput rather than by the spec sheet: the A·1 lesson, closed loop.